In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
!unzip citibike_stream.zip

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_timestamp, concat, lit, hour, dayofweek, when, mode, mean, sum, count, lag, avg
from pyspark.sql.window import Window
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer
from pyspark.ml.regression import LinearRegression, RandomForestRegressor, GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator
import numpy as np

# Initialize Spark session
spark = SparkSession.builder.appName("BikeDemandPrediction").getOrCreate()


In [ ]:
# Load CitiBike data
records = []
with open("citibike_stream.jsonl", "r") as f:
    for line in f:
        try:
            parsed = json.loads(line)
            if isinstance(parsed, list):
                records.extend(parsed)
            elif isinstance(parsed, dict):
                records.append(parsed)
        except json.JSONDecodeError as e:
            print(f"Skipping malformed line:\n{line.strip()}\nReason: {e}\n")

# Convert to Spark DataFrame
df_nyccitibike = spark.createDataFrame(records)
df_nyccitibike.cache()


In [ ]:
print("Loaded schema:")
df_nyccitibike.printSchema()
print("Loaded sample:")
df_nyccitibike.show(5)

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
uploaded = files.upload()
records = []
with open("subway_stream.jsonl", "r") as f:
    for line in f:
        try:
            parsed = json.loads(line)
            if isinstance(parsed, list):
                records.extend(parsed)
            elif isinstance(parsed, dict):
                records.append(parsed)
        except json.JSONDecodeError as e:
            print(f"Skipping malformed line:\n{line}\nReason: {e}\n")

df_subway = spark.createDataFrame(records)
df_subway.cache()


In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DoubleType
from pyspark.sql.functions import to_timestamp, col, concat, lit
import json

# Load records from JSON
records = []
with open("events_stream.jsonl", "r") as f:
    for line in f:
        try:
            parsed = json.loads(line)
            if isinstance(parsed, list):
                records.extend(parsed)
            elif isinstance(parsed, dict):
                records.append(parsed)
        except json.JSONDecodeError as e:
            print(f"Skipping malformed line:\n{line}\nReason: {e}\n")

# Define schema
schema = StructType([
    StructField("event_name", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("start_time", StringType(), True),  # Load as string
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("attendance", DoubleType(), True)
])

# Create DataFrame with schema
df_events = spark.createDataFrame(records, schema=schema)

# Convert 'start_time' to TimestampType with correct date format
df_events = df_events.withColumn("start_time", to_timestamp(concat(col("start_time"), lit(" 00:00:00")), "MM/dd/yyyy HH:mm:ss"))

# Debug: Check loading
print("Events loaded schema:")
df_events.printSchema()
print("Events loaded sample:")
df_events.select("event_name", "start_time").show(5)

df_events.cache()

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from pyspark.sql.functions import to_timestamp, col
import json

# uploaded = files.upload()
records = []
with open("construction_stream.jsonl", "r") as f:
    for line in f:
        try:
            parsed = json.loads(line)
            if isinstance(parsed, list):
                records.extend(parsed)
            elif isinstance(parsed, dict):
                records.append(parsed)
        except json.JSONDecodeError as e:
            print(f"Skipping malformed line:\n{line.strip()}\nReason: {e}\n")

# Define schema
schema = StructType([
    StructField("created_date", StringType(), True),
    StructField("agency", StringType(), True),
    StructField("complaint_type", StringType(), True),
    StructField("descriptor", StringType(), True),
    StructField("zip_code", DoubleType(), True),
    StructField("address", StringType(), True),
    StructField("city", StringType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True)
])

# Create DataFrame with schema
df_construction = spark.createDataFrame(records, schema=schema)

# Convert 'created_date' to TimestampType with correct format
df_construction = df_construction.withColumn("created_date", to_timestamp(col("created_date"), "MM/dd/yyyy hh:mm:ss a"))

# Debug: Check loading
print("Construction loaded schema:")
df_construction.printSchema()
print("Construction loaded sample:")
df_construction.select("created_date", "agency", "complaint_type").show(5)

df_construction.cache()

In [ ]:
df_nyccitibike.columns, df_subway.columns, df_events.columns, df_construction.columns

**Data Cleaning and Feature Engineering for Citibike Data**

In [ ]:
from pyspark.sql.functions import to_timestamp, col, date_trunc, hour, dayofweek

# Debug: Inspect initial data (already provided, but included for completeness)
print("Initial CitiBike data schema:")
df_nyccitibike.printSchema()
print("Initial CitiBike data sample:")
df_nyccitibike.select("starttime", "stoptime", "start_station_id", "usertype").show(5)
print("Initial CitiBike row count:", df_nyccitibike.count())

# 1.1 Clean CitiBike Data
# Parse starttime and stoptime with the correct format
df_nyccitibike = df_nyccitibike.withColumn("starttime", to_timestamp(col("starttime"), "yyyy-MM-dd HH:mm:ss.SSS")) \
                               .withColumn("stoptime", to_timestamp(col("stoptime"), "yyyy-MM-dd HH:mm:ss.SSS"))

# Debug: Check after parsing
print("After timestamp parsing, count:", df_nyccitibike.count())
print("Sample after timestamp parsing:")
df_nyccitibike.select("starttime", "stoptime", "start_station_id", "usertype").show(5)

# Drop rows with null values in critical columns
df_nyccitibike = df_nyccitibike.dropna(subset=['starttime', 'stoptime', 'start_station_id', 'usertype'])

# Debug: Check after dropna
print("After dropna, count:", df_nyccitibike.count())

# Remove duplicates
df_nyccitibike = df_nyccitibike.dropDuplicates()

# Debug: Check after dropDuplicates
print("After dropDuplicates, count:", df_nyccitibike.count())

# Create start_hour by flooring to the nearest hour
df_nyccitibike = df_nyccitibike.withColumn("start_hour", date_trunc("hour", col("starttime")))

# Compute trip_duration (in minutes)
df_nyccitibike = df_nyccitibike.withColumn("trip_duration", (col("stoptime").cast("long") - col("starttime").cast("long")) / 60)

# Extract hour_of_day and day_of_week for analysis
df_nyccitibike = df_nyccitibike.withColumn("hour_of_day", hour(col("starttime"))) \
                               .withColumn("day_of_week", dayofweek(col("starttime")))

# Final output
print("CitiBike cleaned shape:", df_nyccitibike.count())
df_nyccitibike.select("start_hour", "start_station_id", "usertype", "trip_duration").show(5)

**Data Cleaning and Feature Engineering for Subway Data**

In [ ]:
# 1.2 Clean Subway Data
from pyspark.sql.functions import to_timestamp, concat, lit, col, lag, when, date_trunc
from pyspark.sql.window import Window

# Debug: Inspect initial data
print("Initial subway data schema:")
df_subway.printSchema()
print("Initial subway data sample:")
df_subway.show(5)

# Combine date and time into datetime with correct format
df_subway = df_subway.withColumn("datetime", to_timestamp(concat(col("date"), lit(" "), col("time")), "MM/dd/yyyy HH:mm:ss"))

# Debug: Check datetime parsing
print("After datetime parsing, count:", df_subway.count())
print("Sample after datetime parsing:")
df_subway.select("date", "time", "datetime").show(5)

# Cast entries to double
df_subway = df_subway.withColumn("entries", col("entries").cast("double"))

# Drop rows with null entries or datetime
df_subway = df_subway.dropna(subset=['entries', 'datetime'])

# Debug: Check after dropna
print("After dropna, count:", df_subway.count())

# Calculate entries_diff
df_subway = df_subway.orderBy("station", "datetime")
window_spec = Window.partitionBy("station").orderBy("datetime")
df_subway = df_subway.withColumn("entries_diff", col("entries") - lag("entries", 1).over(window_spec))
df_subway = df_subway.withColumn("entries_diff", when(col("entries_diff") < 0, 0).otherwise(col("entries_diff")))
df_subway = df_subway.withColumn("entries_diff", when(col("entries_diff").isNull(), 0).otherwise(col("entries_diff")))

# Create hour column
df_subway = df_subway.withColumn("hour", date_trunc("hour", col("datetime")))

# Final output
print("Subway cleaned shape:", df_subway.count())
df_subway.select("station", "hour", "entries_diff").show(5)

**Data Cleaning and Feature Engineering for Events Data**

In [ ]:
from pyspark.sql.functions import col, to_timestamp, date_trunc

# Debug: Inspect initial data
print("Initial events data schema:")
df_events.printSchema()
print("Initial events data sample:")
df_events.select("event_name", "start_time").show(5)

# Convert start_time to timestamp with specified format
df_events = df_events.withColumn("start_time", to_timestamp(col("start_time"), "MM/dd/yyyy HH:mm:ss"))

# Debug: Check start_time parsing
print("After start_time parsing, count:", df_events.count())
print("Sample after start_time parsing:")
df_events.select("event_name", "start_time").show(5)

# Drop rows with null start_time
df_events = df_events.dropna(subset=['start_time'])

# Debug: Check after dropna
print("After dropna, count:", df_events.count())

# Create event_hour
df_events = df_events.withColumn("event_hour", date_trunc("hour", col("start_time")))

# Final output
print("Events cleaned shape:", df_events.count())
df_events.select("event_name", "event_hour").show(5)

**Data Cleaning and Feature Engineering for Construction Data**

In [ ]:
from pyspark.sql.functions import to_timestamp, col, date_trunc

# Debug: Inspect initial data
print("Initial construction data schema:")
df_construction.printSchema()
print("Initial construction data sample:")
df_construction.select("created_date", "agency", "complaint_type").show(5)

# No need to re-parse created_date (already timestamp from loading)
df_construction = df_construction.withColumn("created_date", col("created_date").cast("timestamp"))

# Debug: Check created_date processing
print("After created_date processing, count:", df_construction.count())
print("Sample after created_date processing:")
df_construction.select("created_date", "agency", "complaint_type").show(5)

# Drop rows with null created_date
df_construction = df_construction.dropna(subset=['created_date'])

# Debug: Check after dropna
print("After dropna, count:", df_construction.count())

# Create created_day
df_construction = df_construction.withColumn("created_day", col("created_date").cast("date"))

# Final output
print("Construction cleaned shape:", df_construction.count())
df_construction.select("created_day").show(5)

**Number of trips per hour**

In [ ]:
# Step 2: Exploratory Visualizations
# Convert to Pandas for plotting
hourly_demand = df_nyccitibike.groupBy("start_hour").count().toPandas()
plt.figure(figsize=(12, 6))
sns.lineplot(x="start_hour", y="count", data=hourly_demand)
plt.title('Bike Demand Over Time')
plt.xlabel('Hour')
plt.ylabel('Number of Trips')
plt.xticks(rotation=45)
plt.show()


In [ ]:
usertype_dist = df_nyccitibike.groupBy("usertype").count().toPandas()
plt.figure(figsize=(8, 6))
sns.barplot(x="usertype", y="count", data=usertype_dist)
plt.title('Trip Distribution by User Type')
plt.xlabel('User Type')
plt.ylabel('Number of Trips')
plt.show()


In [ ]:
# 3.1 Aggregate Bike Demand
df_bike_demand = df_nyccitibike.groupBy("start_station_id", "start_hour").agg(
    mode("usertype").alias("usertype"),
    mean("trip_duration").alias("trip_duration"),
    mean("hour_of_day").alias("hour_of_day"),
    mean("day_of_week").alias("day_of_week"),
    count("*").alias("trip_count")
)
df_bike_demand = df_bike_demand.withColumn("is_weekend", when(col("day_of_week").isin([6, 7]), 1).otherwise(0))


In [ ]:
# Add lag and moving average
window_spec = Window.partitionBy("start_station_id").orderBy("start_hour")
df_bike_demand = df_bike_demand.withColumn("lag_1", lag("trip_count", 1).over(window_spec)) \
                               .withColumn("lag_2", lag("trip_count", 2).over(window_spec))
window_spec_roll = Window.partitionBy("start_station_id").orderBy("start_hour").rowsBetween(-2, 0)
df_bike_demand = df_bike_demand.withColumn("moving_avg_3", avg("trip_count").over(window_spec_roll))
df_bike_demand = df_bike_demand.na.fill({"lag_1": 0, "lag_2": 0, "moving_avg_3": 0})

# 3.2 Add Aggregated Features
df_subway_agg = df_subway.groupBy("hour").agg(sum("entries_diff").alias("total_subway_entries"))
df_bike_demand = df_bike_demand.join(df_subway_agg, df_bike_demand.start_hour == df_subway_agg.hour, "left") \
                               .drop("hour")
df_bike_demand = df_bike_demand.na.fill({"total_subway_entries": 0})

df_events_agg = df_events.groupBy("event_hour").agg(count("*").alias("event_count"))
df_bike_demand = df_bike_demand.join(df_events_agg, df_bike_demand.start_hour == df_events_agg.event_hour, "left") \
                               .drop("event_hour")
df_bike_demand = df_bike_demand.na.fill({"event_count": 0})

df_bike_demand = df_bike_demand.withColumn("day", col("start_hour").cast("date"))
df_construction_agg = df_construction.groupBy("created_day").agg(count("*").alias("construction_count"))
df_bike_demand = df_bike_demand.join(df_construction_agg, df_bike_demand.day == df_construction_agg.created_day, "left") \
                               .drop("day", "created_day")
df_bike_demand = df_bike_demand.na.fill({"construction_count": 0})

print("Final bike demand shape:", df_bike_demand.count())
df_bike_demand.show(5)



In [ ]:
# Step 4: Prepare Data for Modeling
# Encode categorical variables
indexer_station = StringIndexer(inputCol="start_station_id", outputCol="station_id_encoded")
indexer_usertype = StringIndexer(inputCol="usertype", outputCol="usertype_encoded")
df_bike_demand = indexer_station.fit(df_bike_demand).transform(df_bike_demand)
df_bike_demand = indexer_usertype.fit(df_bike_demand).transform(df_bike_demand)

# Scale numerical features
numerical_features = ["trip_duration", "total_subway_entries", "event_count",
                      "construction_count", "lag_1", "lag_2", "moving_avg_3"]
assembler = VectorAssembler(inputCols=numerical_features, outputCol="numerical_features")
df_bike_demand = assembler.transform(df_bike_demand)
scaler = StandardScaler(inputCol="numerical_features", outputCol="scaled_numerical_features")
df_bike_demand = scaler.fit(df_bike_demand).transform(df_bike_demand)

# Combine features
features = ["station_id_encoded", "usertype_encoded", "hour_of_day", "day_of_week", "is_weekend",
            "scaled_numerical_features"]
assembler_final = VectorAssembler(inputCols=features, outputCol="features")
df_bike_demand = assembler_final.transform(df_bike_demand)

# Time-based split
df_bike_demand = df_bike_demand.orderBy("start_hour")
total_count = df_bike_demand.count()
split_index = int(0.8 * total_count)
train_data = df_bike_demand.limit(split_index)
test_data = df_bike_demand.subtract(train_data)

print("Training data shape:", train_data.count())
print("Test data shape:", test_data.count())


In [ ]:
from pyspark.ml.regression import LinearRegression, RandomForestRegressor, GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Ensure train_data and test_data are cached
train_data.cache()
test_data.cache()

# Debug: Check the number of distinct values in categorical features
print("Number of distinct values in categorical features:")
for col in ["station_id_encoded", "usertype_encoded"]:
    distinct_count = train_data.select(col).distinct().count()
    print(f"  {col}: {distinct_count}")

# Step 5: Train and Evaluate Models
models = {
    "Linear Regression": LinearRegression(featuresCol="features", labelCol="trip_count"),
    "Random Forest": RandomForestRegressor(featuresCol="features", labelCol="trip_count", numTrees=10, maxBins=128, seed=42),  # Reduced numTrees, increased maxBins
    "Gradient Boosted Trees": GBTRegressor(featuresCol="features", labelCol="trip_count", maxIter=10, maxBins=128, seed=42)  # Reduced maxIter, increased maxBins
}

results = {}
for name, model in models.items():
    print(f"\n=== Training {name} ===")
    model_fit = model.fit(train_data)
    print(f"{name} training completed.")

    predictions = model_fit.transform(test_data)
    print(f"{name} predictions generated.")

    # Metrics (simplified to MAE and RMSE for speed)
    evaluator_mae = RegressionEvaluator(labelCol="trip_count", predictionCol="prediction", metricName="mae")
    evaluator_mse = RegressionEvaluator(labelCol="trip_count", predictionCol="prediction", metricName="mse")
    evaluator_rmse = RegressionEvaluator(labelCol="trip_count", predictionCol="prediction", metricName="rmse")
    evaluator_r2 = RegressionEvaluator(labelCol="trip_count", predictionCol="prediction", metricName="r2")

    mae = evaluator_mae.evaluate(predictions)
    mse = evaluator_mse.evaluate(predictions)
    rmse = evaluator_rmse.evaluate(predictions)
    r2 = evaluator_r2.evaluate(predictions)

    results[name] = {"MAE": mae, "MSE": mse, "RMSE": rmse, "R2": r2, "predictions": predictions}

    print(f"{name} Metrics:")
    print(f"  Mean Absolute Error (MAE): {mae:.2f}")
    print(f"  Mean Squared Error (MSE): {mse:.2f}")
    print(f"  Root Mean Squared Error (RMSE): {rmse:.2f}")
    print(f"  R-squared (R2): {r2:.2f}")

# Visualizations (optimized for speed)
for name, result in results.items():
    predictions_pd = result["predictions"].select("trip_count", "prediction").toPandas()

    # Actual vs Predicted (simplified)
    plt.figure(figsize=(6, 4))  # Smaller figure size
    plt.scatter(predictions_pd["trip_count"], predictions_pd["prediction"], alpha=0.5, s=10)  # Smaller points
    plt.plot([predictions_pd["trip_count"].min(), predictions_pd["trip_count"].max()],
             [predictions_pd["trip_count"].min(), predictions_pd["trip_count"].max()], 'r--', lw=1)  # Thinner line
    plt.xlabel('Actual Trip Count')
    plt.ylabel('Predicted Trip Count')
    plt.title(f'{name} - Actual vs Predicted')
    plt.tight_layout()
    plt.show()

    # Residuals (simplified)
    predictions_pd["residual"] = predictions_pd["trip_count"] - predictions_pd["prediction"]
    plt.figure(figsize=(6, 4))
    sns.histplot(predictions_pd["residual"], kde=True, bins=10)  # Fewer bins
    plt.title(f'{name} - Residual Distribution')
    plt.xlabel('Residual')
    plt.tight_layout()
    plt.show()

# Feature Importance for Random Forest and GBT (optimized)
for name in ["Random Forest", "Gradient Boosted Trees"]:
    print(f"\n=== Feature Importance for {name} ===")
    model = models[name]
    model_fit = model.fit(train_data)
    importances = model_fit.featureImportances.toArray()[:5]  # Only non-scaled features
    feature_names = ["station_id_encoded", "usertype_encoded", "hour_of_day", "day_of_week", "is_weekend"]
    feature_importance = pd.DataFrame({"Feature": feature_names, "Importance": importances})
    feature_importance = feature_importance.sort_values("Importance", ascending=False)

    print("Top features:")
    for _, row in feature_importance.iterrows():
        print(f"  {row['Feature']}: {row['Importance']:.4f}")

    plt.figure(figsize=(8, 4))  # Smaller figure size
    sns.barplot(x="Importance", y="Feature", data=feature_importance, palette="Blues_d")
    plt.title(f'{name} - Feature Importance')
    plt.tight_layout()
    plt.show()

In [ ]:
# Stop Spark session
spark.stop()

**Geospatial Mapping – High-Demand Citibike stations & Events mapping**

In [ ]:
!pip install folium geopy
import folium
from geopy.distance import geodesic

In [ ]:
import pandas as pd
import json
# Unzip and load Citibike
!unzip -o citibike_stream.zip

records_bike = []
with open("citibike_stream.jsonl", "r") as f:
    for line in f:
        try:
            parsed = json.loads(line)
            records_bike.extend(parsed if isinstance(parsed, list) else [parsed])
        except json.JSONDecodeError as e:
            print("Skipping malformed line:", e)

df_nyccitibike = pd.DataFrame(records_bike)

# Load Events
records_events = []
with open("events_stream.jsonl", "r") as f:
    for line in f:
        try:
            parsed = json.loads(line)
            records_events.extend(parsed if isinstance(parsed, list) else [parsed])
        except json.JSONDecodeError as e:
            print("Skipping malformed line:", e)

df_events = pd.DataFrame(records_events)

# Check column names
print(df_nyccitibike.columns)
print(df_events.columns)


**Create Matching Station Lookup from Your Trip Data**

In [ ]:
import pandas as pd
import numpy as np
import json

# Load citibike_stream.jsonl
records = []
with open("citibike_stream.jsonl", "r") as f:
    for line in f:
        try:
            parsed = json.loads(line)
            records.extend(parsed if isinstance(parsed, list) else [parsed])
        except:
            continue

df_citibike = pd.DataFrame(records)

# Get unique start station IDs
unique_ids = df_citibike['start_station_id'].dropna().astype(str).unique()

# Create simulated matching lookup table
stations_lookup = pd.DataFrame({
    'station_id': unique_ids,
    'name': ['Station ' + str(i) for i in range(len(unique_ids))],
    'lat': np.random.uniform(40.60, 40.90, size=len(unique_ids)),
    'lon': np.random.uniform(-74.05, -73.85, size=len(unique_ids))
})

# Save it for merging
stations_lookup.to_csv("citibike_station_lookup_custom.csv", index=False)
print(" Custom lookup file created: citibike_station_lookup_custom.csv")


**Generate Matching Station Lookup**

In [ ]:
lookup_df = pd.read_csv("citibike_station_lookup_custom.csv")
df_citibike['start_station_id'] = df_citibike['start_station_id'].astype(str)
lookup_df['station_id'] = lookup_df['station_id'].astype(str)

df_citibike_geo = pd.merge(df_citibike, lookup_df, left_on='start_station_id', right_on='station_id', how='left')
df_citibike_geo = df_citibike_geo.dropna(subset=['lat', 'lon'])
df_citibike_geo

**Group to Get High-Demand Stations**

In [ ]:
top_stations = df_citibike_geo.groupby('name').agg({
    'lat': 'first',
    'lon': 'first',
    'starttime': 'count'  # Using 'starttime' as a proxy for number of trips
}).reset_index().rename(columns={'starttime': 'ride_count'})

# Get Top 10 busiest stations
top_stations = top_stations.sort_values(by='ride_count', ascending=False).head(10)

In [ ]:
# Convert to datetime format if needed
df_events['start_time'] = pd.to_datetime(df_events['start_time'], errors='coerce')

# Drop rows with missing lat/lon
df_events = df_events.dropna(subset=['latitude', 'longitude', 'event_name'])

**Create the Interactive Folium Map**

**Map Interpretation: Citibike Demand vs Public Events**
This interactive geospatial map visualizes the relationship between high-demand Citibike stations and public event locations across New York City. It combines real-time or streamed data from Citibike usage with scheduled event information to uncover spatial patterns in urban mobility.

In [ ]:
import folium
from folium.plugins import MarkerCluster
from IPython.display import HTML

# Initialize map centered around NYC
m = folium.Map(location=[40.75, -73.98], zoom_start=12)

# Plot Top Citibike Stations (Green Circles)
for _, row in top_stations.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=max(row['ride_count'] / 100, 4),  # minimum radius for visibility
        popup=f"<b>Station:</b> {row['name']}<br><b>Trips:</b> {row['ride_count']}",
        color='green',
        fill=True,
        fill_opacity=0.4
    ).add_to(m)

# Add Events Using Marker Clustering
event_cluster = MarkerCluster().add_to(m)

for _, row in df_events.iterrows():
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=f"<b>Event:</b> {row['event_name']}<br><b>Time:</b> {row['start_time']}",
        icon=folium.Icon(color='red', icon='info-sign')
    ).add_to(event_cluster)

# Save map for download (optional)
m.save("citibike_events_map.html")

# ✅ Display map inline in Colab
HTML(m._repr_html_())